![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 8 -- Lab 2: Homework Steam Game Prices

Homework. This is a real Olympiad task: the Romanian AI Olympiad's example problem asks for the price of a game on the Steam store from its reviews, score, popularity, genres and release date. You get a training table with prices and a test table without them, and you hand in one prediction per test game. The score is the mean absolute error (MAE): below 6.6 dollars is full marks, below 7.0 is 50 of 60 points, below 8 is 40, below 9 is 30.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** `dataset_train.csv` (3,000 games with `Price`) and `dataset_eval.csv` (600 games without it) from the ONIA problem-proposal repository (`github.com/Olimpiada-AI/Propunere-probleme-si-solutii`), built from the Kaggle Steam Games Dataset. The setup cell reads both straight from GitHub into `train` and `test`.

---
# Setup

In [ ]:
# pandas, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
BASE = "https://raw.githubusercontent.com/Olimpiada-AI/Propunere-probleme-si-solutii/main/Dataset%20si%20evaluator/Dataset/"
train = pd.read_csv(BASE + "dataset_train.csv")
test = pd.read_csv(BASE + "dataset_eval.csv")
print(train.shape, test.shape)

In [ ]:
def report(y_true, y_pred):
    """The three regression scores of the Metrics slide, printed on one line."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"MAE {mae:.2f}   RMSE {rmse:.2f}   R2 {r2:.3f}")
    return mae, rmse, r2

---
# Part 1 -- Read the task and the data

The *Olympiad Practice* slides: name the task type, find the target, find which columns are already numbers.

## Task 1: Columns and target

Display `train.head()` and `train.dtypes`. Which column is missing from `test`? Which columns are numbers already, and which are text that hides a number (`Estimated owners`, `Release date`) or a list (`Genres`)?

In [ ]:
# Your code here


## Task 2: The target's shape

Print `train["Price"].describe()` and draw a histogram of `Price` with 30 bins, a title and axis labels. Most games cost between 5 and 20 dollars; a few cost 60 or more.

In [ ]:
# Your code here


## Task 3: The baseline everyone must beat

Split `train` with `train_test_split(train, test_size=0.2, random_state=42)` into `tr` and `va`. Predict the mean price of `tr` for every row of `va` and compute the MAE against `va["Price"]`. Store it in `baseline_mae` (expected about 8.49).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(baseline_mae - 8.49) < 0.05, 'baseline_mae: MAE of predicting the training mean for every validation row'
print('Task 3 passed')

---
# Part 2 -- Features from the columns that are not numbers yet

Write one function `add_features(d)` that builds the model's input table from a raw table `d`, so that the exact same code runs on `train` and on `test`. Each task adds a column to it.

## Task 4: Numbers as they are, in a function

Write `add_features(d)` that returns a new DataFrame with the columns `Metacritic score`, `Positive`, `Negative` copied from `d`. Then write `fit_and_score(cols)` that splits `train` (same split as Task 3), builds `add_features(tr)` and `add_features(va)`, keeps the columns `cols` of each with `.reindex(columns=cols, fill_value=0)` (a column that a split happens to lack is filled with zeros; Task 7 shows why), fits a `LinearRegression` on the training part, predicts the validation part and calls `report`. Run it with the three columns. Expected MAE about 7.85.

In [ ]:
# Your code here


## Task 5: Owners: a number hidden in text

`Estimated owners` looks like `"20000 - 50000"`. Add to `add_features` a column `owners` equal to the midpoint of the two numbers: split the text on `" - "`, convert both parts to float, average them. Check that `add_features(train)["owners"][0]` is 150000.0, then run `fit_and_score` with `owners` added.

Syntax hint (the shape of the call, not the answer):

```python
parts = d["text_col"].str.split(" - ")     # parts.str[0] is the left piece, parts.str[1] the right
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert add_features(train)['owners'][0] == 150000.0, 'owners: midpoint of the two numbers in the range text'
print('Task 5 passed')

## Task 6: Release year: does EDA say it matters?

Add a column `year` to `add_features` with `pd.to_datetime(d["Release date"], format="mixed").dt.year`. Before modelling, print the mean price per year for 2015 onwards with `groupby` (expected: about 14.6 in 2015 rising to about 29 in 2022). Then run `fit_and_score` with `year` added. Expected MAE about 7.01.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert add_features(train)['year'][0] == 2016, 'year: pd.to_datetime(..., format="mixed").dt.year; row 0 was released in 2016'
print('Task 6 passed')

## Task 7: Genres: one-hot encoding of a list

`Genres` holds several labels per game, like `"Action,Indie"`. `d["Genres"].str.get_dummies(sep=",")` gives one 0/1 column per genre (16 columns on `train`). Print the mean price of Indie games against the rest (`train.groupby(genres["Indie"])["Price"].mean()`, expected 14.27 against 19.07). Then add the genre columns inside `add_features` with `pd.concat([out, d["Genres"].str.get_dummies(sep=",")], axis=1)` and run `fit_and_score(base + genre_cols)` where `genre_cols = list(genres.columns)`. Two rare genres never occur in the validation split; without the `reindex` in `fit_and_score` this call would fail with a `KeyError`. Expected MAE about 6.89.

Syntax hint (the shape of the call, not the answer):

```python
dummies = d["list_col"].str.get_dummies(sep=",")
```

In [ ]:
# Your code here


## Task 8: Two features from EDA: log reviews and positive share

Review counts go from 0 to almost a million; a linear model cannot use such a spread well. Add `log_positive = np.log1p(d["Positive"])` and `positive_share = d["Positive"] / (d["Positive"] + d["Negative"] + 1)` to `add_features`. Run `fit_and_score` with `["Metacritic score", "log_positive", "positive_share", "owners", "year"] + genre_cols` and store the result as `best_cols`. Expected MAE about 6.77: the 50-point band on the validation split.

In [ ]:
# Your code here


---
# Part 3 -- The submission

The Olympiad way: fit the final model on all 3,000 training games, predict the 600 test games, write the file in the required format, and check the file before handing it in.

## Task 9: The test table must have the same columns

Build `X_test = add_features(test)`. Three genres of the training table never occur in the test table, so `X_test` lacks their columns. Fix it with `X_test = X_test.reindex(columns=add_features(train).columns, fill_value=0)` and print `X_test.shape` (expected `(600, 23)`).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert X_test.shape == (600, 23) and X_test.isna().sum().sum() == 0, 'X_test: add_features(test) reindexed to the training columns with fill_value=0'
print('Task 9 passed')

## Task 10: Fit on everything, predict, write submission.csv

Fit a `LinearRegression` on `add_features(train)[best_cols]` and `train["Price"]`, predict `X_test[best_cols]`, and save a DataFrame with the single column `Price` to `submission.csv` with `index=False`. Read the file back and print its shape (expected `(600, 1)`) and `describe()`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pd.read_csv('submission.csv').shape == (600, 1), 'submission.csv: one column named Price, 600 rows, index=False'
print('Task 10 passed')

## Task 11: A sanity check on the predictions

Count how many predicted prices are below 0. A linear model can output negative dollars. Replace them with 0 using `clip(lower=0)`, save the file again, and explain in a comment why this cannot make the MAE worse.

In [ ]:
# Your code here


---
## Task 12: Reflect

List the MAE after Tasks 3, 4, 5, 6, 7 and 8. Which change bought the most? Which change bought nothing, and what does that tell you about that column? Why did the genres need `get_dummies` with `sep=","` instead of a 0-to-15 code? Why did we fit the final model on all 3,000 rows instead of the 2,400 used for scoring?

*Your answers here*

---
## Conclusion

- **Reading a task**: numeric target, MAE score, so linear regression first and a mean-price baseline to beat.
- **Features from text**: the owners range became a midpoint, the date became a year, the genre list became one-hot columns, all inside one function used on train and test alike.
- **EDA guided the features**: the price-by-year table justified `year`, the Indie-versus-rest means justified the genres, the review spread justified `log1p`.
- **The submission**: same columns on test, fit on all training rows, a file in the required format, checked before handing in.

---

Made By **Sattam Altwaim**